# Golden set, round 2: five more targets for the reviewers

**Why.** Round 1 scored 36 comps for two targets (PCTY, WEN). Two targets is the biggest limit on every comparison in `similarity_methods.ipynb`. Round 2 adds five targets from the ten showcase (business-structure pilot) targets, about 65 pairs, and then we adjust from there.

**What this notebook does:**
1. **Builds the pool.** For each round-2 target: every comp that any of the six methods in `similarity_methods.ipynb` lists (v0 one-way top 10, B0 mutual, B1, B2, B3 = method 1 / the v1 baseline, method 2). Pairs round 1 already scored are left out. WEN's 2 unscored pairs (HQI, PLNT) are added.
2. **Writes one blind sheet** with a score / reason / notes column set for each of three reviewers (A, B, C, named in `reviewers.csv`). It is never overwritten once it exists.
3. **Once the filled sheet is back:** checks it, applies the agreed reviewer rule, and shows the hit and rank charts for every method on all seven reviewed targets.

The reviewer guide (rubric, the multi-business rule, how to fill the sheet) is [`v1/GOLDEN_SET_ROUND2.md`](../GOLDEN_SET_ROUND2.md).

**No API calls and no multiples.** The comp lists are read from a pinned `similarity_methods` run, not recomputed.

## 0. Inputs

In [1]:
import json
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data_manifest.json").exists())
sys.path.insert(0, str(ROOT / "v1" / "scripts"))
import facets as fc                     # also puts website_scrape/ and src/ on the path
import golden_set as gs
import similarity_eval as se
from fill_facets import DEFAULT_OUT as FACETS_OUT, output_paths
from project_data import OUTPUTS_LOCAL

pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 220)
RUN = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
V1 = OUTPUTS_LOCAL / "v1"

SIMILARITY_RUN = V1 / "similarity_methods" / "20261104T075557Z"   # pinned: the comp lists and rankings this round labels
ROUND2_TARGETS = ["CSL", "AFG", "AZZ", "BBCP", "BUKS"]          # 3 single-business + the 2 simplest multi-business targets
ROUND1_TARGETS = ["PCTY", "WEN"]
REVIEWERS = ["reviewer_a", "reviewer_b", "reviewer_c"]           # names go in labels/reviewers.csv
OUT = V1 / "golden_set_showcase"                                 # fixed folder: the sheet must stay put
LABELS_DIR = OUT / "labels"
SHEET = LABELS_DIR / "round2_sheet.csv"                          # blank, blind; never overwritten
FILLED = Path(os.environ.get("ROUND2_LABELS", LABELS_DIR / "round2_filled.csv"))   # the filled sheet, exported as CSV
EVAL_OUT = Path(os.environ.get("ROUND2_EVAL_OUT", OUT / "evaluation" / RUN))

sim_info = json.loads((SIMILARITY_RUN / "run.json").read_text(encoding="utf-8"))
universe = pd.read_csv(V1 / "universe" / sim_info["universe_run"] / "universe.csv", dtype=str)
info = universe.set_index("IQ_COMPANY_ID")
ticker = info["target_ciq_ticker"]
id_of = dict(zip(universe["target_ciq_ticker"], universe["IQ_COMPANY_ID"]))
facets = pd.read_csv(output_paths(FACETS_OUT).tables / "facets.csv", dtype=str).fillna("")
facets = facets[(facets["prompt_version"] == fc.PROMPT_VERSION) & (facets["llm_status"] == "ok")]
lists = pd.read_csv(SIMILARITY_RUN / "comp_lists.csv", dtype={"target": str, "comp": str})
lists = lists[lists["sample"] == "training"]
rankings = pd.read_csv(SIMILARITY_RUN / "rankings_showcase.csv.gz", dtype={"target": str, "comp": str})
METHODS = list(dict.fromkeys(lists["method"]))
round1 = json.loads((V1 / "golden_set/labels/reviewer_labels_20261028.json").read_text(encoding="utf-8"))
round1_labels = pd.DataFrame(round1["labels"])
scopes = json.loads((V1 / "business_structure_refinement/split_classification/refined_scopes.json").read_text(encoding="utf-8"))
print(f"comp lists from {SIMILARITY_RUN.name} ({len(METHODS)} methods); round 1: {len(round1_labels)} pairs "
      f"(fetched {round1['fetched_at_utc']}); round 2 targets: {', '.join(ROUND2_TARGETS)}")

comp lists from 20261104T075557Z (6 methods); round 1: 36 pairs (fetched 2026-10-28T11:25:10.429Z); round 2 targets: CSL, AFG, AZZ, BBCP, BUKS


## 1. The round-2 targets

Three run one business (CSL, AFG, AZZ) and two run two (BBCP, BUKS), according to the source-cited business split in `outputs-local/v1/business_structure_refinement/` (Eric's work). The split does **not** create new rows: each company keeps one ID and one facet profile. The scope below is shown to the reviewers beside each target.

In [2]:
def scope(t):
    s = scopes.get(t, {})
    return pd.Series({"company": info.loc[id_of[t], "target_company_name"],
                      "SIC industry": info.loc[id_of[t], "target_primary_sic_industry"],
                      "structure": s.get("decision", "not in the split pilot"),
                      "business groups": len(s.get("groups", [])) or np.nan,
                      "open review flags": ", ".join(s.get("review_flags", [])),
                      "scope": s.get("scope_note", "").removeprefix("Current company scope: ")})

targets_view = pd.DataFrame({t: scope(t) for t in ROUND2_TARGETS}).T
targets_view

,company,SIC industry,structure,business groups,open review flags,scope
CSL,Carlisle Companies Incorporated,Asphalt Paving And Roofing Materials,one_business,1,,"Building envelope products and systems. Completed exits, excluded ..."
AFG,"American Financial Group, Inc.","Fire, Marine, and Casualty Insurance",one_business,1,,Specialty commercial property and casualty insurance. Completed ex...
AZZ,AZZ Inc.,"Coating, Engraving, And Allied Services",one_business,1,,Hot-dip galvanizing and related metal finishing + Coil coating and...
BBCP,"Concrete Pumping Holdings, Inc.",Construction Special Trade Contractors,multiple_businesses,2,B1|B3,Concrete pumping and placement services + Concrete washout waste m...
BUKS,Butler National Corporation,Management Services,multiple_businesses,2,"B1|B2, B1|B3, B2|B3","Aircraft engineering, modifications and avionics services + Commer..."


## 2. The pool: every comp any method lists

Union of the six methods' lists for each round-2 target, minus pairs round 1 scored, plus WEN's two pending pairs. Which methods listed each pair is kept in `pool_with_methods.csv` and **not** shown to reviewers.

In [3]:
reviewed = set(zip(round1_labels["target_id"], round1_labels["comp_id"]))
wanted = [id_of[t] for t in ROUND2_TARGETS + ["WEN"]]
pool_rows = lists[lists["target"].isin(wanted)]
pool_rows = pool_rows[[(t, c) not in reviewed for t, c in zip(pool_rows["target"], pool_rows["comp"])]]
pool = (pool_rows.groupby(["target", "comp"])["method"].agg(lambda m: " | ".join(dict.fromkeys(m))).reset_index()
        .rename(columns={"target": "target_id", "comp": "comp_id", "method": "listed by"}))
pool["methods listing it"] = pool["listed by"].str.count(r"\|") + 1
pool.insert(0, "target", pool["target_id"].map(ticker))
pool.insert(1, "comp", pool["comp_id"].map(ticker))
counts = pool.groupby("target").agg(pairs=("comp", "size"), listed_by_every_method=("methods listing it", lambda s: int((s == len(METHODS)).sum())),
                                    listed_by_one_method=("methods listing it", lambda s: int((s == 1).sum())))
print(f"{len(pool)} pairs to score")
counts.reindex(ROUND2_TARGETS + ["WEN"])

67 pairs to score


,pairs,listed_by_every_method,listed_by_one_method
target,,,
CSL,18,4,3
AFG,11,9,0
AZZ,11,1,6
BBCP,14,0,5
BUKS,11,4,2
WEN,2,0,0


## 3. The sheet

Same blind layout as round 1 (`golden_set.labeler_sheet`): target and comp side by side, in shuffled order, with no rank, method, similarity or multiple.
- **Columns shown:** the comp's ticker, name, SIC industry, website, offering description and customer segment, plus the target's **business scope** (section 1).
- **Pair names:** `CSL-R2-01`, … ("R2" = round 2).
- **Reviewer columns:** `score_`, `reason_if_0_` and `notes_` for each of `reviewer_a`, `reviewer_b` and `reviewer_c`.
- **Files written:** `reviewers.csv` holds the names, and the guide is in `v1/GOLDEN_SET_ROUND2.md`. Neither file is overwritten once it exists.

In [4]:
b1 = facets[facets["business_id"] == "B1"].set_index("IQ_COMPANY_ID")
shown_comp = pd.DataFrame({"ticker": ticker, "company": info["target_company_name"],
                           "sic_industry": info["target_primary_sic_industry"], "website": info["target_ciq_website"],
                           "offering": b1["offering_description"].reindex(info.index).fillna(""),
                           "customers": b1["customer_segment"].reindex(info.index).fillna("")})
shown_target = shown_comp.assign(business_scope=[scope(t)["scope"] if t in scopes else "" for t in shown_comp["ticker"]])
sheet = se.reviewer_sheet(pool[gs.KEY], {"target": shown_target, "comp": shown_comp}, REVIEWERS, seed=2026, prefix="R2")

LABELS_DIR.mkdir(parents=True, exist_ok=True)
if SHEET.exists():
    print(f"kept the existing sheet: {SHEET.relative_to(ROOT)} (never overwritten)")
else:
    sheet.to_csv(SHEET, index=False, encoding="utf-8-sig")        # the BOM lets Excel and Sheets read accents
    print(f"wrote the blank sheet: {SHEET.relative_to(ROOT)} ({len(sheet)} pairs)")
if not (LABELS_DIR / "reviewers.csv").exists():
    pd.DataFrame({"slot": REVIEWERS, "name": ""}).to_csv(LABELS_DIR / "reviewers.csv", index=False)
pool.to_csv(LABELS_DIR / "pool_with_methods.csv", index=False)     # for us, not the reviewers
sheet[["pair", "target_ticker", "comp_ticker", "comp_company", "comp_sic_industry"]].head(8)

kept the existing sheet: outputs-local\v1\golden_set_showcase\labels\round2_sheet.csv (never overwritten)


,pair,target_ticker,comp_ticker,comp_company,comp_sic_industry
0,WEN-R2-01,WEN,PLNT,"Planet Fitness, Inc.",Membership Sports and Recreation Clubs
1,WEN-R2-02,WEN,HQI,"HireQuest, Inc.",Help Supply Services
2,AZZ-R2-01,AZZ,IESC,"IES Holdings, Inc.",Electrical Work
3,AZZ-R2-02,AZZ,FERG,Ferguson Enterprises Inc.,"Hardware, And Plumbing And Heating Equipment"
4,AZZ-R2-03,AZZ,POWL,"Powell Industries, Inc.",Switchgear and Switchboard Apparatus
5,AZZ-R2-04,AZZ,CSW,"CSW Industrials, Inc.","Special Industry Machinery, not elsewhere classified"
6,AZZ-R2-05,AZZ,EML,The Eastern Company,"Cutlery, Handtools, And General Hardware"
7,AZZ-R2-06,AZZ,PPIH,"Perma-Pipe International Holdings, Inc.",Miscellaneous Fabricated Metal Products


## 4. Scores (once the filled sheet is back)

**Where the filled sheet goes:** export it as CSV to `outputs-local/v1/golden_set_showcase/labels/round2_filled.csv`, keeping the same columns, then re-run from here. Until then the cells below only report that they are waiting.

**Rule for using the reviewers' scores** (the same rule as round 1, agreed 2026-11-03):
1. **Source:** only the three reviewers' scores. Grades: 2 = direct comp, 1 = adjacent, 0 = not a comp. No LLM verdict enters this set.
2. **Main set (majority):** a pair's grade is the one at least 2 of the 3 reviewers gave. Pairs with three different grades are left out and listed as "no majority".
3. **Strict check (unanimous):** the pairs all three graded the same, reported separately.
4. **Unlabeled comps:** a comp in a method's list that no reviewer scored is *unlabeled*, never 0.
5. **Metrics:** precision@10 and NDCG@10 on the labeled comps of each list; the good and direct comps found, out of those reviewed; the non-comps listed.

In [5]:
filled = pd.read_csv(FILLED, dtype=str, encoding="utf-8-sig").fillna("") if FILLED.exists() else None
if filled is None:
    print(f"waiting for the filled sheet: {FILLED}")
    problems = pd.DataFrame()
else:
    problems = se.check_reviewer_sheet(filled, REVIEWERS)
    print(f"{len(filled)} pairs read from {FILLED.name}; {len(problems)} problems")
    if set(map(tuple, filled[gs.KEY].to_numpy())) != set(map(tuple, sheet[gs.KEY].to_numpy())):
        print("WARNING: the filled sheet's pairs differ from the pool")
READY = filled is not None and problems.empty
problems.head(20)

waiting for the filled sheet: C:\Users\micha\Qual2Cast\outputs-local\v1\golden_set_showcase\labels\round2_filled.csv


""


In [6]:
if READY:
    majority2, unanimous2, no_majority2 = se.reviewer_sets(filled, REVIEWERS)
    majority1, unanimous1, no_majority1 = se.reviewer_sets(round1_labels)       # round 1: riya, barbara, matthew
    majority = pd.concat([majority1, majority2], ignore_index=True)
    unanimous = pd.concat([unanimous1, unanimous2], ignore_index=True)
    no_majority = pd.concat([no_majority1[["target", "comp"]], no_majority2[["target", "comp"]]], ignore_index=True)
    TARGETS = ROUND1_TARGETS + ROUND2_TARGETS
    print(f"majority set: round 1 {len(majority1)} + round 2 {len(majority2)} pairs; unanimous {len(unanimous)}; "
          f"no majority {len(no_majority)}")
    display(majority.assign(target=majority["target"].map(ticker)).groupby(["target", "grade"]).size().unstack(fill_value=0)
            .reindex(TARGETS))
else:
    print("waiting for the filled sheet")

waiting for the filled sheet


### Hits by method, all seven reviewed targets

Same chart as `similarity_methods.ipynb`, one panel per target: each bar is a method's list, coloured by the majority grade:
- **dark blue:** direct comp (2);
- **light blue:** adjacent (1);
- **gray:** no majority, or not reviewed;
- **red:** not a comp (0).

The text after each bar counts the good comps found, out of those reviewed, and the non-comps listed. The table after the chart sums over all seven targets.

In [7]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

SURFACE, INK, INK2, MUTED, GRID, BASE, BAND = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#f0efec"
DIRECT, ADJACENT, NEUTRAL, NONCOMP = "#1c5cab", "#6da7ec", "#898781", "#e34948"   # validated in similarity_methods
plt.rcParams.update({"font.family": ["Segoe UI", "DejaVu Sans"], "font.size": 9, "figure.facecolor": SURFACE,
                     "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE, "axes.edgecolor": BASE,
                     "axes.labelcolor": INK2, "text.color": INK, "xtick.color": INK2, "ytick.color": INK2,
                     "axes.spines.top": False, "axes.spines.right": False, "axes.titlesize": 10, "axes.titleweight": "bold"})
def short(m):
    # the labels similarity_methods.ipynb uses
    return ("B1 none missing" if m.startswith("B1") else "B2 + info weights" if m.startswith("B2")
            else "method 1 (B3, v1 baseline)" if "method 1" in m else "method 2 (FAME)" if "method 2" in m else m)


SHORT = {m: short(m) for m in METHODS}
KINDS = [("2", "direct comp (2)", DIRECT), ("1", "adjacent (1)", ADJACENT),
         ("?", "no majority, or not reviewed", NEUTRAL), ("0", "not a comp (0)", NONCOMP)]

if READY:
    EVAL_OUT.mkdir(parents=True, exist_ok=True)
    grade_of = {(r.target, r.comp): str(r.grade) for r in majority.itertuples()}
    fig, axes = plt.subplots(4, 2, figsize=(12, 13), sharex=True)
    fig.subplots_adjust(left=0.17, right=0.99, top=0.92, bottom=0.04, hspace=0.45, wspace=0.55)
    for ax, t in zip(axes.ravel(), TARGETS):
        tid = id_of[t]
        good_total = int((majority.loc[majority["target"] == tid, "grade"] >= 1).sum())
        for k, m in enumerate(METHODS):
            comps = lists.loc[(lists["method"] == m) & (lists["target"] == tid), "comp"]
            kinds = pd.Series([grade_of.get((tid, c), "?") for c in comps]).value_counts()
            left = 0
            for code_, _, color in KINDS:
                w = kinds.get(code_, 0)
                if w:
                    ax.barh(k, w - 0.06, left=left, height=0.62, color=color)
                    left += w
            ax.text(len(comps) + 0.3, k, f"{int(kinds.get('2', 0) + kinds.get('1', 0))}/{good_total} · {int(kinds.get('0', 0))}",
                    va="center", fontsize=7.5, color=INK2)
        ax.set_yticks(range(len(METHODS)), [SHORT[m] for m in METHODS], fontsize=8)
        ax.set_ylim(len(METHODS) - 0.5, -0.5)
        ax.set_xlim(0, 13.5)
        ax.set_xticks(range(0, 11, 2))
        ax.grid(axis="x", color=GRID, lw=0.8)
        ax.set_axisbelow(True)
        ax.spines["left"].set_visible(False)
        ax.tick_params(axis="y", length=0)
        ax.set_title(f"{t} ({'round 1' if t in ROUND1_TARGETS else 'round 2'})", loc="left")
    for ax in axes.ravel()[len(TARGETS):]:
        ax.axis("off")
    fig.legend(handles=[Patch(color=c, label=l) for _, l, c in KINDS], loc="upper left", bbox_to_anchor=(0.17, 0.965),
               ncol=4, frameon=False, fontsize=8.5)
    fig.suptitle("What each method lists for the reviewed targets (text: good found / reviewed · non-comps listed)",
                 x=0.17, ha="left", y=0.99, fontsize=11, fontweight="bold")
    fig.savefig(EVAL_OUT / "hits_by_method.png", dpi=150)
    plt.show()

    rows = []
    for m in METHODS:
        mm = se.list_metrics(lists[lists["method"] == m], majority)
        rows.append({"method": m, "targets": len(mm), "good found": int(mm["good listed"].sum()),
                     "good reviewed": int(mm["good total"].sum()), "direct found": int(mm["direct listed"].sum()),
                     "direct reviewed": int(mm["direct total"].sum()), "non-comps listed": int(mm["non-comps listed"].sum()),
                     "precision@10 (mean)": mm["precision@10"].mean(), "NDCG@10 (mean)": mm["NDCG@10"].mean(),
                     "labeled share (mean)": mm["labeled share"].mean()})
    summary = pd.DataFrame(rows).set_index("method")
    summary.to_csv(EVAL_OUT / "summary_by_method.csv")
    display(summary.round(2))
else:
    print("waiting for the filled sheet")

waiting for the filled sheet


### Where every reviewed comp ranks, per round-2 target

The same chart as `similarity_methods.ipynb`. Each line is one reviewed comp, followed from v0 to method 2. The y-axis is its rank among the 372 other training firms.
- **Line colour:** blue for good comps (dark = direct, light = adjacent), gray for non-comps.
- **Filled dot:** in that method's list. A **red dot** is a non-comp that got listed.
- **Hollow blue dot:** a good comp the method missed.

In [8]:
if READY:
    LINE_OF = {"2": DIRECT, "1": ADJACENT, "0": BASE}
    x = np.arange(len(METHODS))
    fig, axes = plt.subplots(len(ROUND2_TARGETS), 1, figsize=(10, 5.2 * len(ROUND2_TARGETS)))
    fig.subplots_adjust(left=0.1, right=0.89, top=0.97, bottom=0.03, hspace=0.3)
    for ax, t in zip(axes, ROUND2_TARGETS):
        tid = id_of[t]
        rev = majority[majority["target"] == tid].assign(grade=lambda d: d["grade"].astype(str))
        rk = {m: rankings[(rankings["method"] == m) & (rankings["target"] == tid)].set_index("comp")["rank"] for m in METHODS}
        ymax = 1
        ax.axhspan(0.5, 10.5, color=BAND, lw=0, zorder=0)
        for c, g in sorted(zip(rev["comp"], rev["grade"]), key=lambda p: p[1]):
            r = np.array([rk[m].get(c, np.nan) for m in METHODS], dtype=float)
            ymax = max(ymax, np.nanmax(r))
            good = g in ("1", "2")
            ax.plot(x, r, color=LINE_OF[g], lw=2 if good else 1, zorder=3 if good else 1)
            for xi, ri, m in zip(x, r, METHODS):
                on = ((lists["method"] == m) & (lists["target"] == tid) & (lists["comp"] == c)).any()
                if on:
                    ax.plot(xi, ri, "o", ms=8, mfc=NONCOMP if g == "0" else LINE_OF[g], mec=SURFACE, mew=1.5, zorder=5)
                elif good:
                    ax.plot(xi, ri, "o", ms=7, mfc=SURFACE, mec=LINE_OF[g], mew=1.8, zorder=4)
            ax.text(-0.14, r[0], ticker[c], ha="right", va="center", fontsize=7.5, color=INK if good else MUTED)
            if good:
                ax.text(x[-1] + 0.14, r[-1], ticker[c], ha="left", va="center", fontsize=7.5, color=INK)
        ax.set_xlim(-0.65, len(METHODS) - 0.35)
        ax.set_ylim(min(ymax, 60) + 1, 0.2)
        ax.set_xticks(x, [SHORT[m].replace(" (", "\n(") for m in METHODS], fontsize=8)
        ax.set_ylabel("rank (1 = most similar)")
        ax.spines["left"].set_visible(False)
        ax.spines["bottom"].set_visible(False)
        ax.tick_params(length=0)
        ax.set_title(t, loc="left")
    fig.savefig(EVAL_OUT / "ranks_by_method.png", dpi=150)
    plt.show()
else:
    print("waiting for the filled sheet")

waiting for the filled sheet


## 5. For Eric: the multi-business rule (stub)

**Proposed rule (in the guide):** grade a comp against the **target as a whole**. A comp that matches only one of the target's businesses gets **at most 1 (adjacent)**, and the reviewer names that business in the notes. A 2 still requires the same product **and** the same customers, as in the rubric.

**The round-2 multi-business targets:**
- **BBCP (Concrete Pumping):** concrete pumping and placement with concrete washout waste management; temporary power and site infrastructure. The split has an open review flag (`B1|B3`): the pair evidence conflicts.
- **BUKS (Butler National):** aircraft engineering, modifications and avionics, with electronics manufacturing and precision machining; and gaming facility management (a casino). Three review flags are open, on the aerospace sub-lines.

**TODO for Eric** (please edit this cell and the "Several businesses" section of `v1/GOLDEN_SET_ROUND2.md`):
- [ ] Confirm or replace the rule above, before the sheet goes out.
- [ ] Check that the business scopes shown in the sheet (section 1) are current, and resolve or note the open review flags for BBCP and BUKS.
- [ ] Decide whether a comp that matches only the **minor** business (e.g. a casino operator for BUKS) should be capped at 1 or scored 0.
- [ ] Optional, for the analysis: whether multi-business targets should be reported separately in section 4, by tagging BBCP and BUKS.

## 6. Status

In [9]:
print(f"sheet: {SHEET.relative_to(ROOT)} ({'present' if SHEET.exists() else 'missing'}), {len(sheet)} pairs")
print(f"filled sheet: {FILLED} ({'present' if FILLED.exists() else 'not yet'})")
print(f"reviewers: {LABELS_DIR.relative_to(ROOT) / 'reviewers.csv'}")
print("evaluation:", EVAL_OUT if READY else "not run (waiting for the filled sheet)")

sheet: outputs-local\v1\golden_set_showcase\labels\round2_sheet.csv (present), 67 pairs
filled sheet: C:\Users\micha\Qual2Cast\outputs-local\v1\golden_set_showcase\labels\round2_filled.csv (not yet)
reviewers: outputs-local\v1\golden_set_showcase\labels\reviewers.csv
evaluation: not run (waiting for the filled sheet)
